# Chapter 4 &mdash; Basics of Designing a DFA

**Concept 9 of the Chapter 4 decomposition:** *Basics of Designing a DFA*

A four-step recipe: list examples, fix $\varepsilon$, name by convention, make every state answer every symbol.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Designing-A-DFA/Concept-Designing-A-DFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Design is programming with **two constructs**: test the symbol, then *go to* a state.

The recipe:

1. **write down accepted and rejected strings** &mdash; pin the spec before drawing;
2. **if $\varepsilon$ is accepted, make the initial state final**;
3. **name states by the I/F/IF convention**;
4. **make every state respond to every symbol** &mdash; the step beginners skip.

## 2. Definitions

### Step 1-2: the examples decide the initial state

Target: strings over $\{0,1\}$ containing `01`.

In [ ]:
positives = ['01', '001', '010', '1101', '0101']
negatives = ['', '0', '1', '00', '111', '10']
print("epsilon accepted? ", '' in positives, " -> initial state is NOT final")

### Steps 3-4: name by convention, cover every symbol

In [ ]:
has01 = md2mc('''DFA
I  : 0 -> S0
I  : 1 -> I
S0 : 0 -> S0
S0 : 1 -> F
F  : 0 -> F
F  : 1 -> F
''')
print("states :", sorted(has01["Q"]), " final:", sorted(has01["F"]))
assert len(has01["Delta"]) == len(has01["Q"]) * len(has01["Sigma"])

## 3. Tests

Every positive example is accepted.

In [ ]:
for s in positives: print("%-8r accepted? %s" % (s, accepts_dfa(has01, s)))
assert all(accepts_dfa(has01, s) for s in positives)

Every negative example is rejected.

In [ ]:
for s in negatives: print("%-8r accepted? %s" % (s, accepts_dfa(has01, s)))
assert not any(accepts_dfa(has01, s) for s in negatives)

And it agrees with the specification on everything short.

In [ ]:
from itertools import product
ok = all(accepts_dfa(has01, ''.join(p)) == ('01' in ''.join(p))
         for k in range(9) for p in product('01', repeat=k))
print("agrees with ('01' in s) up to length 8 :", ok)
assert ok

## 4. Animation

Follow the design: `I` waits, `S0` has seen a `0`, `F` has seen `01` and never leaves.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(has01, FuseEdges=True)

## 5. Exercises


1. Design a DFA for "contains `010`". How many states, and why more than three?
2. What changes if $\varepsilon$ *is* to be accepted?
3. Skip step 4 deliberately &mdash; leave out one transition. What does `md2mc` do?

In [ ]:
# Your work for the exercises above.